In [2]:
import os

from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv

In [3]:
mmlu = load_dataset("cais/mmlu", "high_school_macroeconomics", split="test").select(range(100))
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
samsum = load_dataset("knkarthick/samsum", split="test")

In [7]:
mmlu.features

{'question': Value('string'),
 'subject': Value('string'),
 'choices': List(Value('string')),
 'answer': ClassLabel(names=['A', 'B', 'C', 'D'])}

In [22]:
gsm8k

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})

In [24]:
samsum

Dataset({
    features: ['id', 'dialogue', 'summary'],
    num_rows: 819
})

In [22]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [44]:
inputs = f"""
For the following question, provide a very semi-long persona string, containing two sentences, that is specific to the broader subject and does not contain knowledge needed to solve the question itself, starting with "You are a macroeconomist":\n\n

{mmlu[0]['question']}\n\n

(A) {mmlu[0]['choices'][0]}\n
(B) {mmlu[0]['choices'][1]}\n
(C) {mmlu[0]['choices'][2]}\n
(D) {mmlu[0]['choices'][3]}\n

"""

In [47]:
response = client.responses.create(
    model="gpt-5-nano",
    input=inputs,
    prompt_cache_key="2"
)

In [51]:
print(response)

Response(id='resp_68d13e39570081a1a2f92a2f344d73890aac4bf170c34b52', created_at=1758543417.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_68d13e3aa90081a19c15873d4b798b680aac4bf170c34b52', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_68d13e3daa3c81a1803ccb97ffacd0f10aac4bf170c34b52', content=[ResponseOutputText(annotations=[], text='You are a macroeconomist who studies how fiscal policy, monetary policy, and external shocks interact to determine GDP, inflation, and unemployment across business cycles. You favor intuitive explanations grounded in aggregate demand and supply analysis and prefer data-driven insights, while avoiding giving away the solution to a specific problem and staying focused on broad principles.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], par

In [48]:
print(response.output_text)

You are a macroeconomist who studies how fiscal policy, monetary policy, and external shocks interact to determine GDP, inflation, and unemployment across business cycles. You favor intuitive explanations grounded in aggregate demand and supply analysis and prefer data-driven insights, while avoiding giving away the solution to a specific problem and staying focused on broad principles.


In [15]:
mmlu[0]

{'question': 'Suppose that an expansionary fiscal policy leads to a large increase in real output and a small increase in the price level. From this it can be inferred that',
 'subject': 'high_school_macroeconomics',
 'choices': ['inflation had already impacted the economy before the fiscal stimulus.',
  'the economy initially had some unemployed resources.',
  'aggregate supply decreased.',
  'aggregate demand is steeply sloped.'],
 'answer': 1}

In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet("evaluation/data/mmlu.parquet")

In [4]:
df

,question,subject,choices,answer,base_persona,static_short_persona,static_long_persona,dynamic_short_persona,dynamic_long_persona
0,Find the degree for the given field extension ...,abstract_algebra,"[0, 4, 2, 6]",1,None,None,None,None,None
1,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",abstract_algebra,"[8, 2, 24, 120]",2,None,None,None,None,None
2,Find all zeros in the indicated finite field o...,abstract_algebra,"[0, 1, 0,1, 0,4]",3,None,None,None,None,None
3,Statement 1 | A factor group of a non-Abelian ...,abstract_algebra,"[True, True, False, False, True, False, False,...",1,None,None,None,None,None
4,Find the product of the given polynomials in t...,abstract_algebra,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",1,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...
14037,What has been a central focus of religious tra...,world_religions,"[Peace and harmony, Power and influence, Truth...",0,None,None,None,None,None
14038,To whom did ordinary folk appeal during a dro...,world_religions,"[The Buddha, Laozi, The Queen Mother of the We...",2,None,None,None,None,None
14039,The theological term homoousios means which o...,world_religions,"[of a similar substance, of the same substance...",1,None,None,None,None,None
14040,"According to the Japanese origin myth, who giv...",world_religions,"[Es, Izanagi, Izanami, Kami]",1,None,None,None,None,None


In [6]:
df.to_parquet("test.parquet")

In [5]:
df["static"] = None

In [5]:
df[df["subject"] == "high_school_macroeconomics"]

,question,subject,choices,answer,base_persona,static_short_persona,static_long_persona,dynamic_short_persona,dynamic_long_persona
3838,Suppose that an expansionary fiscal policy lea...,high_school_macroeconomics,[inflation had already impacted the economy be...,1,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
3839,Which of the following is included in U.S. GDP...,high_school_macroeconomics,"[II III and IV only, I and III only, II and IV...",3,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
3840,When both short-run aggregate supply and aggre...,high_school_macroeconomics,"[The price level rises but real GDP falls., Bo...",3,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
3841,Tariffs and quotas,high_school_macroeconomics,"[result in lower domestic prices., sometimes r...",2,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
3842,A likely cause of falling Treasury bond prices...,high_school_macroeconomics,"[expansionary monetary policy., contractionary...",1,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
...,...,...,...,...,...,...,...,...,...
4223,Which of the following is NOT an argument for ...,high_school_macroeconomics,"[To protect infant industry, To promote employ...",2,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
4224,The GDP Deflator differs from the CPI in that ...,high_school_macroeconomics,[is thought to slightly overestimate the infla...,3,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
4225,Business cycles,high_school_macroeconomics,"[occur infrequently in capitalist economies., ...",3,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None
4226,When disposable income increases by $X,high_school_macroeconomics,"[consumption increases by more than $X ., savi...",1,You are a knowledgeable macroeconomist,You are a knowledgeable macroeconomist who exp...,You are a knowledgeable macroeconomist who exp...,None,None


In [47]:
df.loc[df["subject"] == "abstract_algebra", "static"].isnull().all()

np.False_

In [52]:
df.loc[df["subject"] == "abstract_algebra", "static"] = "Hello"

In [45]:
df.loc[0, "static"] = "Hello"

In [54]:
df.loc[df["subject"] == "abstract_algebra", "static"].unique()[0]

'Hello'

In [9]:
df = pd.read_parquet("evaluation/data/gsm8k.parquet")

In [10]:
df.head()

,question,answer
0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...
1,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...
2,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...
3,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t..."


In [11]:
df.to_parquet("test2.parquet")

In [12]:
df = pd.read_parquet("test2.parquet")

In [13]:
df.head()

,question,answer
0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...
1,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...
2,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...
3,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t..."


In [6]:
df = pd.read_parquet("evaluation/data/samsum.parquet")

In [7]:
df.head()

,id,dialogue,summary,base_persona,static_short_persona,static_long_persona,dynamic_short_persona,dynamic_long_persona
0,13862856,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,You are an expert in summarization,"You are an expert in summarization, skilled at...","You are an expert in summarization, skilled at...",None,None
1,13729565,Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...,Eric and Rob are going to watch a stand-up on ...,You are an expert in summarization,"You are an expert in summarization, skilled at...","You are an expert in summarization, skilled at...",None,None
2,13680171,"Lenny: Babe, can you help me with something?\n...",Lenny can't decide which trousers to buy. Bob ...,You are an expert in summarization,"You are an expert in summarization, skilled at...","You are an expert in summarization, skilled at...",None,None
3,13729438,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,You are an expert in summarization,"You are an expert in summarization, skilled at...","You are an expert in summarization, skilled at...",None,None
4,13828600,"Ollie: Hi , are you in Warsaw\nJane: yes, just...",Jane is in Warsaw. Ollie and Jane has a party....,You are an expert in summarization,"You are an expert in summarization, skilled at...","You are an expert in summarization, skilled at...",None,None
